In [ ]:
#!/usr/bin/env python3
"""
================================================================================
CHECK MULTI-SOURCING -- quanto spesso il MILP vero assegna una SKU a piu' DC
================================================================================
Se la risposta e' "spesso", nessuna euristica single/limited-sourcing (buona
o cattiva) potra' mai avvicinarsi all'ottimo per quelle SKU -- il limite e'
strutturale. Se "raramente", il gap XGBoost e' piu' attribuibile alla qualita'
dell'euristica che alla rigidita' strutturale in se'.

File necessari nella stessa cartella:
  - optimality_gap_v2.py
  - check_outliers.py (per solve_free_detailed, column_generation, ecc.)
  - gnn_dataset/
================================================================================
"""
import json, glob, os, time
import numpy as np
import gurobipy as gp
from gurobipy import GRB

import sys
sys.path.insert(0, os.getcwd())
from optimality_gap_v2 import inst_from_json, column_generation, get_time_limit, THREADS_PER_SOLVE, MIP_GAP_TARGET, build_milp


# ============================================================================
# CONFIG -- ADATTA IL PATH
# ============================================================================
DATASET_DIR = r"C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_dataset_v2_TEST"


SAMPLE_IDS = [53, 84, 178, 265, 415, 342, 355, 386, 502, 470, 172, 485]
def solve_free_detailed(inst, warm_start, n_threads, time_limit, mip_gap):
    """Come solve_free originale, ma espone am, dc, n3DP completi."""
    mdl, v, _ = build_milp(inst, n_threads, time_limit, mip_gap)
    ws = warm_start
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    for i in range(nI):
        v['am'][i].Start = float(ws['am'][i]); v['cm'][i].Start = float(1-ws['am'][i])
    for d in range(nD):
        for i in range(nI):
            v['dc'][d,i].Start = float(ws['dc'][d,i])
            v['v'][i,d].Start = float(ws['am'][i] and ws['dc'][d,i])
            v['w'][i,d].Start = float((not ws['am'][i]) and ws['dc'][d,i])
            for c in range(nC): v['y'][d,i,c].Start = float(ws['y'][d,i,c]>0.5)
        v['n3DP'][d].Start = float(ws['n3dp'][d])
    mdl.update(); mdl.optimize()
    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] or mdl.SolCount == 0:
        return None
    return {
        'obj': mdl.ObjVal, 'gap': mdl.MIPGap*100,
        'am': np.array([v['am'][i].X>0.5 for i in range(nI)]),
        'dc': np.array([[v['dc'][d,i].X>0.5 for i in range(nI)] for d in range(nD)]),
        'n3DP': np.array([v['n3DP'][d].X for d in range(nD)]),
    }


if __name__ == '__main__':
    files = sorted(glob.glob(os.path.join(DATASET_DIR, "instance_*.json")))
    id_to_file = {}
    for fp in files:
        with open(fp) as fh:
            d = json.load(fh)
        id_to_file[int(d['instance_id'])] = fp
    print(f"File trovati: {len(files)}, istanze indicizzate: {len(id_to_file)}")

    tot_sku_am, tot_sku_cm = 0, 0
    multi_am, multi_cm = 0, 0
    dettaglio = []

    for iid in SAMPLE_IDS:
        if iid not in id_to_file:
            print(f"#{iid}: non trovato, skip"); continue
        with open(id_to_file[iid]) as fh:
            d = json.load(fh)
        nI, nD, nC = d['nI'], d['nD'], d['nC']
        inst = inst_from_json(d)

        print(f"\n{'='*60}\n#{iid} (nI={nI}, nD={nD}, nC={nC})\n{'='*60}")
        t0 = time.time()
        ws = column_generation(inst, n_threads=THREADS_PER_SOLVE)
        res = solve_free_detailed(inst, ws, n_threads=THREADS_PER_SOLVE,
                                   time_limit=get_time_limit(nI), mip_gap=MIP_GAP_TARGET)
        print(f"  risolto in {time.time()-t0:.1f}s")
        if res is None:
            print("  INFEASIBLE, skip"); continue

        am, dc = res['am'].astype(int), res['dc'].astype(int)  # dc shape (nD, nI)
        n_dc_per_sku = dc.sum(axis=0)  # quanti DC usa ciascuna SKU

        for i in range(nI):
            is_multi = n_dc_per_sku[i] > 1
            if am[i]:
                tot_sku_am += 1
                if is_multi: multi_am += 1
            else:
                tot_sku_cm += 1
                if is_multi: multi_cm += 1

        print(f"  n_DC per SKU: {n_dc_per_sku.tolist()}")
        print(f"  SKU multi-sourced (>1 DC): {(n_dc_per_sku>1).sum()}/{nI}")
        dettaglio.append({
            'instance_id': iid, 'nI': nI, 'nD': nD,
            'n_dc_per_sku': n_dc_per_sku.tolist(), 'am': am.tolist(),
        })

    print(f"\n{'='*60}\nRIEPILOGO CAMPIONE ({len(dettaglio)} istanze)\n{'='*60}")
    print(f"SKU AM totali: {tot_sku_am}  -- multi-sourced: {multi_am} ({100*multi_am/max(tot_sku_am,1):.1f}%)")
    print(f"SKU CM totali: {tot_sku_cm}  -- multi-sourced: {multi_cm} ({100*multi_cm/max(tot_sku_cm,1):.1f}%)")
    print(f"Totale multi-sourced: {multi_am+multi_cm}/{tot_sku_am+tot_sku_cm} "
          f"({100*(multi_am+multi_cm)/max(tot_sku_am+tot_sku_cm,1):.1f}%)")

    with open('multisourcing_check.json', 'w') as f:
        json.dump(dettaglio, f, indent=2, default=str)
    print("\nDettaglio salvato in multisourcing_check.json")

c:\Users\stefa\anaconda3\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


File trovati: 608, istanze indicizzate: 608

#53 (nI=8, nD=2, nC=5)
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
  risolto in 3.8s
  n_DC per SKU: [1, 1, 1, 1, 1, 1, 1, 1]
  SKU multi-sourced (>1 DC): 0/8

#84 (nI=8, nD=2, nC=5)
  risolto in 2.3s
  n_DC per SKU: [1, 1, 1, 1, 2, 2, 2, 2]
  SKU multi-sourced (>1 DC): 4/8

#178 (nI=10, nD=3, nC=5)
  risolto in 35.0s
  n_DC per SKU: [1, 1, 1, 1, 2, 2, 2, 2, 2, 2]
  SKU multi-sourced (>1 DC): 6/10

#265 (nI=10, nD=3, nC=6)
  risolto in 2950.6s
  n_DC per SKU: [1, 1, 1, 2, 2, 2, 2, 2, 3, 3]
  SKU multi-sourced (>1 DC): 7/10

#415 (nI=15, nD=3, nC=6)
  risolto in 177.6s
  n_DC per SKU: [1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 3]
  SKU multi-sourced (>1 DC): 8/15

#342 (nI=12, nD=3, nC=6)
  risolto in 22.1s
  n_DC per SKU: [1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2]
  SKU multi-sourced (>1 DC): 6/12

#355 (nI=12, nD=3, nC=6)
  risolto in 78.6s
  n_DC per SKU: [1,